# บทที่ 1: พื้นฐานของโครงข่ายประสาทเทียม (Neural Network Basics)

ใน Notebook นี้ เราจะเรียนรู้การ implement องค์ประกอบพื้นฐานของ Neural Network ตั้งแต่ฟังก์ชัน Sigmoid, Artificial Neuron, Perceptron ไปจนถึงการแก้ปัญหา XOR ด้วย Multi-Layer Perceptron

## 1. นำเข้าไลบรารีที่จำเป็น (Import Libraries)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

## 2. ฟังก์ชัน Sigmoid และกราฟ

ฟังก์ชัน Sigmoid เป็น activation function ที่แปลงค่า input ให้อยู่ในช่วง (0, 1)

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

In [ ]:
def sigmoid(x):
    """ฟังก์ชัน Sigmoid"""
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """อนุพันธ์ของฟังก์ชัน Sigmoid"""
    s = sigmoid(x)
    return s * (1 - s)

# สร้างข้อมูลสำหรับ plot
x = np.linspace(-10, 10, 100)
y_sig = sigmoid(x)
y_deriv = sigmoid_derivative(x)

# แสดงกราฟ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x, y_sig, 'b-', linewidth=2, label='Sigmoid')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('σ(x)')
axes[0].set_title('ฟังก์ชัน Sigmoid')
axes[0].legend()

axes[1].plot(x, y_deriv, 'r-', linewidth=2, label="Sigmoid'")
axes[1].set_xlabel('x')
axes[1].set_ylabel("σ'(x)")
axes[1].set_title('อนุพันธ์ของ Sigmoid')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. การคำนวณของ Artificial Neuron

Artificial Neuron คำนวณผลลัพธ์จากสมการ:

$$y = \sigma(\sum_{i=1}^{n} w_i x_i + b)$$

In [ ]:
def artificial_neuron(x, weights, bias):
    """
    คำนวณผลลัพธ์ของ Artificial Neuron
    
    Parameters:
    - x: input vector
    - weights: weight vector
    - bias: bias value
    
    Returns:
    - output: ผลลัพธ์หลังผ่าน sigmoid
    """
    z = np.dot(weights, x) + bias
    return sigmoid(z)

# ตัวอย่างการคำนวณ
x = np.array([0.5, 0.3, 0.8])
weights = np.array([0.2, -0.1, 0.4])
bias = 0.1

output = artificial_neuron(x, weights, bias)
print(f"Input: {x}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"Output: {output:.4f}")

## 4. Perceptron และ Perceptron Learning Rule

Perceptron เป็นโมเดล Neural Network ที่ง่ายที่สุด ใช้สำหรับปัญหา Linear Classification

In [ ]:
class Perceptron:
    """
    Perceptron Classifier
    """
    def __init__(self, n_features, learning_rate=0.1, n_epochs=100):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
        self.n_epochs = n_epochs
        
    def activation(self, z):
        """Step function"""
        return 1 if z >= 0 else 0
    
    def predict(self, x):
        """ทำนายผลลัพธ์"""
        z = np.dot(self.weights, x) + self.bias
        return self.activation(z)
    
    def fit(self, X, y):
        """
        เรียนรู้จากข้อมูล
        
        Perceptron Learning Rule:
        w_new = w_old + lr * (target - prediction) * x
        b_new = b_old + lr * (target - prediction)
        """
        for epoch in range(self.n_epochs):
            errors = 0
            for xi, target in zip(X, y):
                prediction = self.predict(xi)
                error = target - prediction
                
                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1
            
            if errors == 0:
                print(f"Converged at epoch {epoch + 1}")
                break
                
        return self

## 5. Logic Gates: AND, OR, XOR

In [ ]:
# ข้อมูลสำหรับ Logic Gates
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# AND Gate
y_and = np.array([0, 0, 0, 1])

# OR Gate
y_or = np.array([0, 1, 1, 1])

# XOR Gate
y_xor = np.array([0, 1, 1, 0])

print("=== AND Gate ===")
perceptron_and = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_and.fit(X, y_and)
print(f"Weights: {perceptron_and.weights}")
print(f"Bias: {perceptron_and.bias}")
for xi, yi in zip(X, y_and):
    print(f"Input: {xi}, Target: {yi}, Prediction: {perceptron_and.predict(xi)}")

print("\n=== OR Gate ===")
perceptron_or = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_or.fit(X, y_or)
print(f"Weights: {perceptron_or.weights}")
print(f"Bias: {perceptron_or.bias}")
for xi, yi in zip(X, y_or):
    print(f"Input: {xi}, Target: {yi}, Prediction: {perceptron_or.predict(xi)}")

## 6. ปัญหา XOR - Perceptron เดี่ยวไม่สามารถแก้ได้

In [ ]:
print("=== XOR Gate (Single Perceptron - จะไม่ converge) ===")
perceptron_xor = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_xor.fit(X, y_xor)
print(f"Weights: {perceptron_xor.weights}")
print(f"Bias: {perceptron_xor.bias}")
print("\nผลการทำนาย:")
for xi, yi in zip(X, y_xor):
    print(f"Input: {xi}, Target: {yi}, Prediction: {perceptron_xor.predict(xi)}")

print("\nสรุป: Perceptron เดี่ยวไม่สามารถแก้ปัญหา XOR ได้ เพราะ XOR ไม่ linearly separable")

## 7. การแสดงภาพ Decision Boundary

In [ ]:
def plot_decision_boundary(perceptron, X, y, title, ax):
    """แสดง Decision Boundary"""
    # สร้าง grid
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    
    # ทำนายทุกจุดใน grid
    Z = np.array([perceptron.predict(np.array([x, y])) 
                  for x, y in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)
    
    # Plot
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=100, edgecolors='black')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_decision_boundary(perceptron_and, X, y_and, 'AND Gate', axes[0])
plot_decision_boundary(perceptron_or, X, y_or, 'OR Gate', axes[1])
plot_decision_boundary(perceptron_xor, X, y_xor, 'XOR Gate (Failed)', axes[2])

plt.tight_layout()
plt.show()

## 8. การแก้ปัญหา XOR ด้วย Multi-Layer Perceptron (MLP)

In [ ]:
class MLP:
    """
    Multi-Layer Perceptron สำหรับแก้ปัญหา XOR
    """
    def __init__(self, n_input, n_hidden, n_output, learning_rate=0.5):
        # Initialize weights
        self.W1 = np.random.randn(n_hidden, n_input) * 0.5
        self.b1 = np.zeros((n_hidden, 1))
        self.W2 = np.random.randn(n_output, n_hidden) * 0.5
        self.b2 = np.zeros((n_output, 1))
        self.lr = learning_rate
        
    def forward(self, x):
        """Forward pass"""
        self.x = x.reshape(-1, 1)
        self.z1 = np.dot(self.W1, self.x) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.W2, self.a1) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2
    
    def backward(self, y):
        """Backward pass"""
        y = y.reshape(-1, 1)
        m = 1  # batch size
        
        # Output layer gradients
        dz2 = self.a2 - y
        dW2 = np.dot(dz2, self.a1.T)
        db2 = dz2
        
        # Hidden layer gradients
        dz1 = np.dot(self.W2.T, dz2) * sigmoid_derivative(self.z1)
        dW1 = np.dot(dz1, self.x.T)
        db1 = dz1
        
        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        
    def train(self, X, y, epochs=10000):
        """Train the network"""
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)
                
    def predict(self, x):
        """Predict"""
        return self.forward(x)[0, 0]

# สร้างและฝึก MLP
mlp = MLP(n_input=2, n_hidden=2, n_output=1, learning_rate=0.5)
mlp.train(X, y_xor, epochs=10000)

print("=== XOR กับ MLP ===")
print(f"W1:\n{mlp.W1}")
print(f"b1:\n{mlp.b1}")
print(f"W2:\n{mlp.W2}")
print(f"b2:\n{mlp.b2}")
print("\nผลการทำนาย:")
for xi, yi in zip(X, y_xor):
    pred = mlp.predict(xi)
    print(f"Input: {xi}, Target: {yi}, Prediction: {pred:.4f} → {1 if pred > 0.5 else 0}")

## 9. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณผลลัพธ์ของ Sigmoid
จงคำนวณค่า sigmoid ของค่าต่อไปนี้:
- σ(0) = ?
- σ(1) = ?
- σ(-1) = ?
- σ(5) = ?

In [ ]:
# เขียนโค้ดคำนวณที่นี่
values = [0, 1, -1, 5]
for v in values:
    print(f"σ({v}) = {sigmoid(v):.4f}")

### แบบฝึกหัดที่ 2: คำนวณผลลัพธ์ของ Neuron
ให้ Neuron มีค่าดังนี้:
- Input: x₁ = 0.5, x₂ = 0.8, x₃ = 0.2
- Weights: w₁ = 0.3, w₂ = -0.2, w₃ = 0.5
- Bias: b = 0.1

จงคำนวณ:
1. ค่า z = Σwᵢxᵢ + b
2. ผลลัพธ์ y = σ(z)

In [ ]:
# เขียนโค้ดคำนวณที่นี่
x = np.array([0.5, 0.8, 0.2])
weights = np.array([0.3, -0.2, 0.5])
bias = 0.1

z = np.dot(weights, x) + bias
y = sigmoid(z)

print(f"z = {z:.4f}")
print(f"y = σ(z) = {y:.4f}")

### แบบฝึกหัดที่ 3: Perceptron Learning
ให้ Perceptron เริ่มต้นด้วย:
- w = [0, 0], b = 0
- Learning rate = 0.1

จงคำนวณค่า w และ b หลังจากเรียนรู้จากตัวอย่าง:
- Input: [1, 1], Target: 1, Prediction: 0

In [ ]:
# เขียนโค้ดคำนวณที่นี่
w = np.array([0.0, 0.0])
b = 0.0
lr = 0.1

x = np.array([1, 1])
target = 1
prediction = 0
error = target - prediction

w_new = w + lr * error * x
b_new = b + lr * error

print(f"ก่อนอัปเดต: w = {w}, b = {b}")
print(f"หลังอัปเดต: w = {w_new}, b = {b_new}")

### แบบฝึกหัดที่ 4: คำนวณจำนวน Parameters
จงคำนวณจำนวน parameters ของ Neural Network ที่มี:
- Input Layer: 4 neurons
- Hidden Layer 1: 8 neurons
- Hidden Layer 2: 4 neurons
- Output Layer: 2 neurons

In [ ]:
# เขียนโค้ดคำนวณที่นี่
def count_parameters(layers):
    """คำนวณจำนวน parameters"""
    total = 0
    for i in range(len(layers) - 1):
        weights = layers[i] * layers[i+1]
        biases = layers[i+1]
        total += weights + biases
        print(f"Layer {i} → {i+1}: weights = {weights}, biases = {biases}")
    return total

layers = [4, 8, 4, 2]
total_params = count_parameters(layers)
print(f"\nTotal parameters: {total_params}")

## บทสรุป

Notebook นี้แสดงให้เห็นว่า:
1. Perceptron เดี่ยวสามารถแก้ปัญหา Linear Classification ได้ (AND, OR)
2. Perceptron เดี่ยวไม่สามารถแก้ปัญหา XOR ได้ เพราะ XOR ไม่ใช่ Linearly Separable
3. Multi-Layer Perceptron (MLP) สามารถแก้ปัญหา XOR ได้

ผู้อ่านสามารถทดลองเปลี่ยนค่า parameters ต่างๆ เพื่อสังเกตผลลัพธ์ที่เปลี่ยนแปลงไป